# Prompt chaining: break one hard prompt into three easy ones

**The Agent Build Log · Build #013 · Prompt Engineering 5/8**

So far the ticket prompt only labelled tickets. A real support tool also pulls out the order details and drafts a
reply that follows company policy. This build does all three jobs **in one prompt**, then as a **chain of three
prompts**, and checks every output against rules a program can verify.

1. [The job and the policy](#1-the-job-and-the-policy)
2. [Three designs](#2-three-designs)
3. [Run them](#3-run-them)
4. [Which policy rules broke](#4-which-policy-rules-broke)
5. [One ticket, three designs](#5-one-ticket-three-designs)
6. [Findings](#6-findings)

**Model:** Claude Haiku 5.5, thinking off. **Needs:** an Anthropic API key. A full run costs about **10 cents**.

In [1]:
import json, os, re, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Literal, Optional

import pandas as pd
from pydantic import BaseModel
from dotenv import load_dotenv
import anthropic

load_dotenv(Path.cwd() / ".env") or load_dotenv(Path.cwd().parents[1] / ".env")
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in a .env file (see README)"
pd.set_option("display.width", 170); pd.set_option("display.max_colwidth", 100)

MODEL = "claude-haiku-5-5"
PRICE = {"input": 0.10, "output": 0.50}
NO_THINKING = {"type": "disabled"}
client = anthropic.Anthropic().with_options(max_retries=6)
TK = json.loads(Path("data/tickets.json").read_text())
TICKETS, FIELDS = TK["tickets"], TK["fields"]

def cost(u):
    return (u.input_tokens * PRICE["input"] + u.output_tokens * PRICE["output"]) / 1e6
print(f"{len(TICKETS)} tickets | {sum(t['refund_requested'] for t in TICKETS)} ask for money back | "
      f"{sum(t['order_id'] is not None for t in TICKETS)} mention an order | {sum(t['label']['priority'] == 'urgent' for t in TICKETS)} urgent")

40 tickets | 9 ask for money back | 14 mention an order | 4 urgent


## 1. The job and the policy

For each ticket: **labels** (category, priority, sentiment, using build #010's rubric v2), **fields** (order id, and
whether the customer asks for money back), and a **reply** that follows five policy rules. Every rule can be checked
by code, so nothing here is graded by opinion:

| Rule | Check |
|---|---|
| Under 80 words | word count |
| Quote the order id if the ticket has one | the id appears in the reply |
| If they ask for money back, say refunds take 5-7 business days (and only then) | "5-7 business days" appears exactly when it should |
| If the ticket is urgent, say it's been escalated to the on-call team (and only then) | "escalat" appears exactly when it should |
| Never mention internal labels | no "priority", "category" or "sentiment" in the reply |

The 40 tickets are build #004's 20 and build #010's 20 fresh ones. Order ids come from #004's labels or the ticket text;
"asks for money back" was labelled by hand before any model run.

In [2]:
ROLE = "You are the support triage lead at an online electronics shop. You sort incoming tickets so the right team picks them up first."
RUBRIC = '''How our support team labels tickets:
Category
- billing: charges, refunds, plans, pricing questions, cancellations
- bug: something in the product is broken or wrong
- account: logging in, security, account settings, users
- shipping: delivery, tracking, damaged or wrong items
- feature_request: ideas for the product, and general feedback or praise
Priority
- urgent: many users blocked or money being lost right now, or a security incident (someone else in the account)
- high: one customer blocked with no workaround (locked out, charged twice for the same order, item lost or damaged,
  a crash on a main screen), or a business deadline at risk
- medium: a real problem with a workaround or a simple fix (wrong amount, coupon, cancellation, late tracking,
  exchange, a non-blocking bug), or a feature they would use right away
- low: questions, "someday" ideas, praise
Sentiment
- negative: frustration, or reports something broken, lost, damaged or blocked for them
- neutral: a calm request or question, including calm billing requests
- positive: praise or thanks is the main point
When two rules disagree:
- A calm request about a billing mistake is neutral, even though something went wrong.
- A bug the customer calls minor or small is still negative: it reports something broken.
- If the customer has a working alternative (another device, the website), it's medium, even if one path is blocked.'''   # build #010's v2

POLICY = '''Reply policy:
- Keep the reply under 80 words.
- If the ticket mentions an order id, include it in the reply.
- If the customer asks for money back, say refunds take 5-7 business days to arrive. Otherwise don't mention refund timelines.
- If the ticket is urgent, say it has been escalated to the on-call team. Otherwise don't say it was escalated.
- Never mention internal labels such as category, priority or sentiment.'''

def check(reply: str, t: dict) -> dict:
    low = reply.lower()
    timeline = bool(re.search(r"5\s*[-–to]+\s*7 business days", low))
    return {
        "under 80 words": len(reply.split()) < 80,
        "quotes order id": t["order_id"] is None or t["order_id"].lower() in low,
        "refund timeline only when asked": timeline == t["refund_requested"],
        "escalation only when urgent": ("escalat" in low) == (t["label"]["priority"] == "urgent"),
        "no internal labels": not re.search(r"\b(priority|category|sentiment)\b", low),
    }
print(check("Thanks! Refunds for ORD-1 take 5-7 business days.", {"order_id": "ORD-1", "refund_requested": True, "label": {"priority": "high"}}))

{'under 80 words': True, 'quotes order id': True, 'refund timeline only when asked': True, 'escalation only when urgent': True, 'no internal labels': True}


## 2. Three designs

- **One prompt:** rubric + policy + ticket in, one structured object out (labels, fields, reply).
- **Chain, code picks the rules:** step 1 labels the ticket, step 2 extracts the fields, then **code** turns those
  answers into the rules that apply ("say refunds take 5-7 business days", "quote ORD-48213") and step 3 writes the reply
  with only those rules.
- **Chain, all rules:** the same three steps, but step 3 gets the whole policy and the step 1-2 answers, and has to work
  out which rules apply itself. This separates the effect of *splitting the prompt* from the effect of *code choosing the rules*.

In [3]:
Category = Literal["billing", "bug", "account", "shipping", "feature_request"]
Priority = Literal["low", "medium", "high", "urgent"]
Sentiment = Literal["negative", "neutral", "positive"]

class Package(BaseModel):
    category: Category
    priority: Priority
    sentiment: Sentiment
    order_id: Optional[str]
    asks_for_money_back: bool
    reply: str

class Labels(BaseModel):
    category: Category
    priority: Priority
    sentiment: Sentiment

class Fields(BaseModel):
    order_id: Optional[str]
    asks_for_money_back: bool

def call(messages, schema=None, system=ROLE, max_tokens=800):
    t0 = time.perf_counter()
    kw = dict(model=MODEL, max_tokens=max_tokens, thinking=NO_THINKING, system=system, messages=messages)
    r = client.messages.parse(output_format=schema, **kw) if schema else client.messages.create(**kw)
    out = r.parsed_output if schema else "".join(b.text for b in r.content if b.type == "text").strip()
    return out, cost(r.usage), time.perf_counter() - t0

def ticket_block(t):
    return f"<ticket>\n{t['text']}\n</ticket>"

def one_prompt(t):
    p, c, s = call([{"role": "user", "content": f"{RUBRIC}\n\n{POLICY}\n\n{ticket_block(t)}\n\n"
                     "Label this ticket, extract its order id (or null) and whether the customer asks for money back, "
                     "and write the reply to the customer following the policy."}], Package)
    return {"labels": {"category": p.category, "priority": p.priority, "sentiment": p.sentiment},
            "order_id": p.order_id, "money_back": p.asks_for_money_back, "reply": p.reply, "cost": c, "seconds": s, "calls": 1}

def steps_1_2(t):
    lab, c1, s1 = call([{"role": "user", "content": f"{RUBRIC}\n\n{ticket_block(t)}\n\nLabel this ticket."}], Labels)
    fld, c2, s2 = call([{"role": "user", "content": f"{ticket_block(t)}\n\nExtract the order id (or null if there is none) "
                         "and whether the customer asks for money back (a refund, or money returned)."}], Fields,
                       system="You extract fields from support tickets.")
    return lab, fld, c1 + c2, s1 + s2

def rules_for(lab, fld):
    rules = ["Keep the reply under 80 words.", "Never mention internal labels such as category, priority or sentiment."]
    if fld.order_id:
        rules.append(f"Mention the order id {fld.order_id}.")
    rules.append("Say refunds take 5-7 business days to arrive." if fld.asks_for_money_back else "Don't mention refund timelines.")
    rules.append("Say it has been escalated to the on-call team." if lab.priority == "urgent" else "Don't say it was escalated.")
    return "\n".join(f"- {r}" for r in rules)

def chain(t, code_picks_rules: bool):
    lab, fld, c12, s12 = steps_1_2(t)
    if code_picks_rules:
        instr = f"Write the reply to this customer. Follow these rules:\n{rules_for(lab, fld)}"
    else:
        instr = (f"{POLICY}\n\nWhat we know about this ticket: priority {lab.priority}, order id {fld.order_id}, "
                 f"asks for money back: {fld.asks_for_money_back}.\n\nWrite the reply to this customer, following the policy.")
    reply, c3, s3 = call([{"role": "user", "content": f"{ticket_block(t)}\n\n{instr}\n\nReply with only the message to the customer."}],
                         system="You write short, friendly replies to support tickets for an online electronics shop.")
    return {"labels": lab.model_dump(), "order_id": fld.order_id, "money_back": fld.asks_for_money_back, "reply": reply,
            "cost": c12 + c3, "seconds": s12 + s3, "calls": 3}

DESIGNS = {"one prompt": one_prompt, "chain, code picks the rules": lambda t: chain(t, True),
           "chain, all rules": lambda t: chain(t, False)}

## 3. Run them

40 tickets × 3 runs × 3 designs. A chain's time is the sum of its three calls (they run one after another).

In [4]:
def run(design, t, run_no):
    o = DESIGNS[design](t)
    chk = check(o["reply"], t)
    oid = (o["order_id"] or "").strip().upper() or None
    return {"design": design, "id": t["id"], "set": t["set"], "run": run_no,
            "labels right": all(o["labels"][f] == t["label"][f] for f in FIELDS),
            "fields right": oid == t["order_id"] and o["money_back"] == t["refund_requested"],
            "reply follows all 5 rules": all(chk.values()), **chk,
            "words": len(o["reply"].split()), "cost": o["cost"], "seconds": o["seconds"], "calls": o["calls"], "reply": o["reply"]}

jobs = [(d, t, r) for d in DESIGNS for t in TICKETS for r in (1, 2, 3)]
with ThreadPoolExecutor(8) as ex:
    runs = pd.DataFrame(list(ex.map(lambda j: run(*j), jobs)))
print(f"{len(runs)} ticket-runs | ${runs['cost'].sum():.4f}")

RULES = ["under 80 words", "quotes order id", "refund timeline only when asked", "escalation only when urgent", "no internal labels"]
summary = runs.groupby("design", sort=False).agg(**{
    "labels right %": ("labels right", lambda s: round(100 * s.mean())),
    "fields right %": ("fields right", lambda s: round(100 * s.mean())),
    "reply follows all 5 rules %": ("reply follows all 5 rules", lambda s: round(100 * s.mean())),
    "everything right %": ("id", lambda s: 0),
    "calls per ticket": ("calls", "first"),
    "median s per ticket": ("seconds", lambda s: round(s.median(), 2)),
    "$ per 1,000 tickets": ("cost", lambda s: round(s.mean() * 1000, 3)),
})
all_ok = runs["labels right"] & runs["fields right"] & runs["reply follows all 5 rules"]
summary["everything right %"] = (100 * all_ok.groupby(runs["design"], sort=False).mean()).round()
summary

360 ticket-runs | $0.0814


,labels right %,fields right %,reply follows all 5 rules %,everything right %,calls per ticket,median s per ticket,"$ per 1,000 tickets"
design,,,,,,,
one prompt,77,100,97,73.0,1,1.42,0.200
"chain, code picks the rules",80,100,93,74.0,3,2.88,0.232
"chain, all rules",82,100,74,57.0,3,2.88,0.246


## 4. Which policy rules broke

Share of replies that passed each rule (120 per design):

In [5]:
rule_table = (100 * runs.groupby("design", sort=False)[RULES].mean()).round().astype(int)
rule_table

,under 80 words,quotes order id,refund timeline only when asked,escalation only when urgent,no internal labels
design,,,,,
one prompt,100,100,98,100,99
"chain, code picks the rules",97,100,98,100,98
"chain, all rules",90,100,98,82,94


In [6]:
# the refund rule, split: replies that should mention 5-7 business days vs replies that shouldn't
r = runs.merge(pd.DataFrame([{"id": t["id"], "asks": t["refund_requested"]} for t in TICKETS]), on="id")
r["mentions timeline"] = r["reply"].str.lower().str.contains(r"5\s*[-–to]+\s*7 business days", regex=True)
pd.crosstab([r["design"], r["asks"]], r["mentions timeline"]).rename(columns={True: "said 5-7 days", False: "didn't"})

mentions timeline                  didn't  said 5-7 days
design                      asks                        
chain, all rules            False      93              0
                            True        3             24
chain, code picks the rules False      93              0
                            True        3             24
one prompt                  False      93              0
                            True        3             24

## 5. One ticket, three designs

Ticket 13 (build #004): billed for two plans, wants one refunded. First run of each design:

In [7]:
t13 = next(t for t in TICKETS if t["id"] == 13)
print("TICKET:", t13["text"], "\nasks for money back:", t13["refund_requested"], "| order id:", t13["order_id"], "\n")
for d in DESIGNS:
    x = runs[(runs.design == d) & (runs.id == 13) & (runs.run == 1)].iloc[0]
    failed = [k for k in RULES if not x[k]]
    print(f"--- {d} | {x['words']} words | failed: {failed or 'none'}\n{x['reply']}\n")

TICKET: I was billed $9.99 this month and also $99 for the annual plan I chose. Only keep the annual one and refund the $9.99. 
asks for money back: True | order id: None 

--- one prompt | 49 words | failed: none
Thanks for reaching out. I can see you were charged both $9.99 for the monthly plan and $99 for the annual plan. We'll keep your annual plan and refund the $9.99 monthly charge. Refunds take 5-7 business days to arrive. Please let us know if you need anything else.

--- chain, code picks the rules | 50 words | failed: none
Hi there,

Thanks for reaching out! We're happy to help. We'll keep your annual plan and refund the $9.99 charge. Please allow 5-7 business days for the refund to appear on your original payment method.

If you have any other questions, just reply to this message.

Best regards,
Customer Support

--- chain, all rules | 65 words | failed: none
Hi there,

Thanks for reaching out, and sorry for the confusion with your billing. We're glad to help. You'll keep y

## 6. Findings

Every number below comes from the cells above; these are the numbers the LinkedIn post uses.

In [8]:
f = {
    "model": MODEL, "sdk": anthropic.__version__, "thinking": "disabled", "cost_total": round(runs["cost"].sum(), 4),
    "summary": summary.reset_index().to_dict("records"), "rules": rule_table.reset_index().to_dict("records"),
    "runs": runs.to_dict("records"),
}
Path("findings.json").write_text(json.dumps(f, indent=2, default=str))
print(json.dumps({k: f[k] for k in ("summary", "rules", "cost_total")}, indent=1, default=str))

{
 "summary": [
  {
   "design": "one prompt",
   "labels right %": 77,
   "fields right %": 100,
   "reply follows all 5 rules %": 97,
   "everything right %": 73.0,
   "calls per ticket": 1,
   "median s per ticket": 1.42,
   "$ per 1,000 tickets": 0.2
  },
  {
   "design": "chain, code picks the rules",
   "labels right %": 80,
   "fields right %": 100,
   "reply follows all 5 rules %": 93,
   "everything right %": 74.0,
   "calls per ticket": 3,
   "median s per ticket": 2.88,
   "$ per 1,000 tickets": 0.232
  },
  {
   "design": "chain, all rules",
   "labels right %": 82,
   "fields right %": 100,
   "reply follows all 5 rules %": 74,
   "everything right %": 57.0,
   "calls per ticket": 3,
   "median s per ticket": 2.88,
   "$ per 1,000 tickets": 0.246
  }
 ],
 "rules": [
  {
   "design": "one prompt",
   "under 80 words": 100,
   "quotes order id": 100,
   "refund timeline only when asked": 98,
   "escalation only when urgent": 100,
   "no internal labels": 99
  },
  {
   "desi